# 축3 자가소비 적합성 분석
## 02. 전처리

### 개요
- 축2 전처리 데이터(742개) 기준으로 축3 필요 변수 추가
- 추가 변수: 운영시간, 자치구, EV 충전소 반경 500m 집계, 자치구별 EV 등록대수

### 데이터 출처
| 변수 | 출처 |
|------|------|
| 운영시간 | OA-13122 원본 CSV |
| 자치구 | 주소에서 파싱 |
| EV 충전소 500m | 한국환경공단 API (서울_EV충전소.csv) |
| EV 등록대수 | 서울시 OA-15640 |

In [1]:
import pandas as pd
import numpy as np

# 축2 전처리 데이터 로드
df = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\parking_preprocessed.csv',
    encoding='utf-8-sig'
)

# OA-13122 원본 로드
parking_raw = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\raw\서울시 공영주차장 안내 정보(원본).csv',
    encoding='cp949'
)

print(f"축2 전처리 데이터: {df.shape}")
print(f"원본 데이터: {parking_raw.shape}")
print(f"\n원본 컬럼: {parking_raw.columns.tolist()}")

축2 전처리 데이터: (742, 9)
원본 데이터: (3696, 39)

원본 컬럼: ['주차장코드', '주차장명', '주소', '주차장 종류', '주차장 종류명', '운영구분', '운영구분명', '전화번호', '주차현황 정보 제공여부', '주차현황 정보 제공여부명', '총 주차면', '유무료구분', '유무료구분명', '야간무료개방여부', '야간무료개방여부명', '평일 운영 시작시각(HHMM)', '평일 운영 종료시각(HHMM)', '주말 운영 시작시각(HHMM)', '주말 운영 종료시각(HHMM)', '공휴일 운영 시작시각(HHMM)', '공휴일 운영 종료시각(HHMM)', '최종데이터 동기화 시간', '토요일 유,무료 구분', '토요일 유,무료 구분명', '공휴일 유,무료 구분', '공휴일 유,무료 구분명', '월 정기권 금액', '노상 주차장 관리그룹번호', '기본 주차 요금', '기본 주차 시간(분 단위)', '추가 단위 요금', '추가 단위 시간(분 단위)', '버스 기본 주차 요금', '버스 기본 주차 시간(분 단위)', '버스 추가 단위 시간(분 단위)', '버스 추가 단위 요금', '일 최대 요금', '위도', '경도']


In [2]:
print(f"전체 행 수: {len(df)}")
print(f"고유 주차장명 수: {df['pklt_nm'].nunique()}")
print(f"중복 주차장명 수: {df['pklt_nm'].duplicated().sum()}")

# 중복 있으면 확인
if df['pklt_nm'].duplicated().sum() > 0:
    print(df[df['pklt_nm'].duplicated(keep=False)][['pklt_nm']].sort_values('pklt_nm'))

전체 행 수: 742
고유 주차장명 수: 741
중복 주차장명 수: 1
           pklt_nm
225  동작대교(위) 공영주차장
226  동작대교(위) 공영주차장


In [3]:
# 동작대교(위) 두 행 비교
print(df[df['pklt_nm'] == '동작대교(위) 공영주차장'].to_string())

           pklt_nm pklt_knd_nm  tpkct  add_crg_std  add_crg_std_filled  is_add_crg_imputed        lat         lot           addr
225  동작대교(위) 공영주차장      노상 주차장    1.0        520.0               520.0               False  37.505355  126.980618  동작구 동작동 316-3
226  동작대교(위) 공영주차장      노상 주차장   51.0        520.0               520.0               False  37.505355  126.980618  동작구 동작동 316-3


### Step 0. 중복 주차장 처리
- 동작대교(위) 공영주차장: 동일 위치에 1면(노상 개별면)과 51면(실제 주차장) 2행 존재
- 1면짜리(tpkct=1) 제거 → 741개로 통일

In [4]:
# 중복 확인
print(f"제거 전: {len(df)}개")
print(df[df['pklt_nm'] == '동작대교(위) 공영주차장'][['pklt_nm', 'tpkct', 'lat', 'lot']])

# 1면짜리 제거 (같은 이름 중 tpkct가 작은 것)
df = df.drop(index=225).reset_index(drop=True)

print(f"\n제거 후: {len(df)}개")
print(f"고유 주차장명: {df['pklt_nm'].nunique()}개")

제거 전: 742개
           pklt_nm  tpkct        lat         lot
225  동작대교(위) 공영주차장    1.0  37.505355  126.980618
226  동작대교(위) 공영주차장   51.0  37.505355  126.980618

제거 후: 741개
고유 주차장명: 741개


### Step 1. 운영시간 추출
- OA-13122 원본에서 평일/주말 운영시간 추출
- HHMM 정수형 → 시간 단위 변환
- 2400(자정) 예외처리
- 주차장명 기준 중복 제거 후 축2 데이터에 조인

In [5]:
import re

parking_raw = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\raw\서울시 공영주차장 안내 정보(원본).csv',
    encoding='cp949'
)

def calc_hours(start, end):
    try:
        s = int(str(int(start)).zfill(4)[:2])
        e = int(str(int(end)).zfill(4)[:2])
        if e == 24:
            e = 24
        hours = e - s
        return hours if hours > 0 else 24
    except:
        return None

parking_raw['평일운영시간'] = parking_raw.apply(
    lambda r: calc_hours(r['평일 운영 시작시각(HHMM)'], r['평일 운영 종료시각(HHMM)']), axis=1)
parking_raw['주말운영시간'] = parking_raw.apply(
    lambda r: calc_hours(r['주말 운영 시작시각(HHMM)'], r['주말 운영 종료시각(HHMM)']), axis=1)

# 주차장명 기준 중복 제거 (운영시간 중앙값)
ops_time = parking_raw.groupby('주차장명')[['평일운영시간', '주말운영시간']].median().reset_index()

print(f"운영시간 추출 완료: {ops_time.shape}")
print(ops_time[['주차장명', '평일운영시간', '주말운영시간']].head(10).to_string())
print(f"결측: 평일 {ops_time['평일운영시간'].isna().sum()}개, 주말 {ops_time['주말운영시간'].isna().sum()}개")

운영시간 추출 완료: (1694, 3)
               주차장명  평일운영시간  주말운영시간
0  (구)동작구청 부설주차장(구)    24.0    24.0
1      (구)북부지법주변(구)    10.0     9.0
2      01-05-023(구)    24.0    24.0
3      01-09-102(구)    24.0    24.0
4      01-09-106(구)    24.0    24.0
5      01-09-113(구)    24.0    24.0
6      01-26-036(구)    24.0    24.0
7      01-26-049(구)    24.0    24.0
8      01-26-052(구)    24.0    24.0
9      01-37-114(구)    24.0    24.0
결측: 평일 0개, 주말 3개


### Step 2. 주차장명 정규화 및 운영시간 조인
- 축2 전처리 데이터(pklt_nm)와 원본(주차장명) 표기 방식이 달라 직접 조인 불가
- 정규화 키 생성 후 매칭
- 정규화: (시)·(구)·괄호 제거, 공백 제거

In [6]:
def normalize_name(name):
    name = str(name)
    name = re.sub(r'\(시\)|\(구\)|舊', '', name)
    name = re.sub(r'\([^)]*\)', '', name)
    name = re.sub(r'\s+', '', name)
    return name.strip()

df['nm_key'] = df['pklt_nm'].apply(normalize_name)
ops_time['nm_key'] = ops_time['주차장명'].apply(normalize_name)

# 중복 제거 후 조인
ops_time_dedup = ops_time.drop_duplicates(subset='nm_key', keep='first')

df_merged = df.merge(
    ops_time_dedup[['nm_key', '평일운영시간', '주말운영시간']],
    on='nm_key', how='left'
)

print(f"조인 후 shape: {df_merged.shape}")
print(f"운영시간 결측: {df_merged['평일운영시간'].isna().sum()}개")
print(f"주말 결측: {df_merged['주말운영시간'].isna().sum()}개")

조인 후 shape: (741, 12)
운영시간 결측: 0개
주말 결측: 2개


In [7]:
print(df_merged[df_merged['주말운영시간'].isna()][['pklt_nm', 'nm_key', '평일운영시간', '주말운영시간']].to_string())

   pklt_nm  nm_key  평일운영시간  주말운영시간
96  구룡산제1호  구룡산제1호    10.0     NaN
97  구룡산제2호  구룡산제2호    10.0     NaN


- 구룡산제1·2호: 원본 자체에 주말 운영시각 없음
- 평일만 운영하는 주차장으로 판단 → 주말운영시간 = 0으로 처리

In [8]:
# 주말 미운영 → 0으로 처리
df_merged['주말운영시간'] = df_merged['주말운영시간'].fillna(0)

# 가중평균 운영시간 (평일×5 + 주말×2) / 7
df_merged['운영시간'] = (df_merged['평일운영시간'] * 5 + df_merged['주말운영시간'] * 2) / 7

print(f"결측 확인: {df_merged['운영시간'].isna().sum()}개")
print(f"\n구룡산 확인:")
print(df_merged[df_merged['pklt_nm'].str.contains('구룡산')][['pklt_nm', '평일운영시간', '주말운영시간', '운영시간']].to_string())
print(f"\n운영시간 분포:")
print(df_merged['운영시간'].describe())

결측 확인: 0개

구룡산 확인:
   pklt_nm  평일운영시간  주말운영시간      운영시간
96  구룡산제1호    10.0     0.0  7.142857
97  구룡산제2호    10.0     0.0  7.142857

운영시간 분포:
count    741.000000
mean      17.902159
std        6.641420
min        7.142857
25%       10.000000
50%       24.000000
75%       24.000000
max       24.000000
Name: 운영시간, dtype: float64


### Step 3. 자치구 추출
- 주소(addr)에서 자치구 파싱

In [ ]:
df_merged['자치구'] = df_merged['addr'].str.extract(r'(\S+구)')

print(f"자치구 결측: {df_merged['자치구'].isna().sum()}개")
print(f"\n자치구 분포:")
print(df_merged['자치구'].value_counts())

자치구 결측: 0개

자치구 분포:
자치구
영등포구    75
양천구     64
강남구     59
강서구     43
송파구     40
금천구     36
서초구     35
중구      32
종로구     29
마포구     28
구로구     26
강북구     26
노원구     25
용산구     25
강동구     24
성동구     23
동대문구    22
동작구     21
도봉구     20
관악구     20
은평구     16
중랑구     15
성북구     13
광진구     13
서대문구    11
Name: count, dtype: int64


#### 자치구 추출 결과
- 741개 전체 결측 없음
- 영등포구(75개), 양천구(64개), 강남구(59개) 순으로 많음
- EDA 결과와 일치

### Step 4. 자치구별 EV 등록대수 조인
- 서울시 자치구별 연료별 자동차 등록현황 (OA-15640, 2026년 3월 기준)
- 전기차만 필터링 후 자치구별 합산

In [10]:
df_car = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\raw\서울시 자치구별 연료별 차종별 용도별 등록현황(26년3월).csv',
    encoding='cp949'
)

# 전기차만 필터링
ev_by_gu = df_car[
    (df_car['연료명'] == '전기') &
    (df_car['시군구명'] != '합 계')
].groupby('시군구명')['계'].sum().reset_index()
ev_by_gu.columns = ['자치구', 'ev_등록대수']

# 조인
df_merged = df_merged.merge(ev_by_gu, on='자치구', how='left')

print(f"결측: {df_merged['ev_등록대수'].isna().sum()}개")
print(f"shape: {df_merged.shape}")
print(df_merged[['pklt_nm', '자치구', 'ev_등록대수']].head(5).to_string())

결측: 0개
shape: (741, 15)
        pklt_nm   자치구  ev_등록대수
0     15-가600구간   강동구     5653
1         21-37   송파구     7851
2           BYC   금천구     2257
3  KBS별관뒤 노상주차장  영등포구     4323
4  KBS별관옆 노상주차장  영등포구     4323


#### EV 등록대수 조인 결과
- 741개 전체 결측 없음
- 자치구별 EV 등록대수 정상 조인

### Step 6. 전처리 완료 데이터 저장
- 축3 분석에 필요한 컬럼만 선택
- 중간 계산용 컬럼(nm_key 등) 제거

In [32]:
cols = [
    'pklt_nm',        # 주차장명
    'pklt_knd_nm',    # 노상/노외
    'lat', 'lot',     # 위경도
    'addr',           # 주소
    '자치구',          # 자치구
    '평일운영시간',     # 평일 운영시간
    '주말운영시간',     # 주말 운영시간
    '운영시간',        # 가중평균 운영시간
    'ev_등록대수',     # 자치구별 EV 등록대수
]

df_axis3 = df_merged[cols].copy()

print(f"최종 shape: {df_axis3.shape}")
print(f"결측 확인:")
print(df_axis3.isna().sum())

df_axis3.to_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\parking_axis3_preprocessed.csv',
    index=False, encoding='utf-8-sig'
)
print("\n저장 완료")

최종 shape: (741, 10)
결측 확인:
pklt_nm        0
pklt_knd_nm    0
lat            0
lot            0
addr           0
자치구            0
평일운영시간         0
주말운영시간         0
운영시간           0
ev_등록대수        0
dtype: int64

저장 완료


### 전처리 최종 결과
- 741개 전체 결측 없음
- 저장: `parking_axis3_preprocessed.csv`

| 컬럼 | 설명 |
|------|------|
| pklt_nm | 주차장명 |
| pklt_knd_nm | 노상/노외 구분 |
| lat, lot | 위경도 |
| addr | 주소 |
| 자치구 | 주소에서 파싱 |
| 평일운영시간 | 평일 운영시간 (시간) |
| 주말운영시간 | 주말 운영시간 (시간) |
| 운영시간 | 가중평균 운영시간 (평일×5 + 주말×2) / 7 |
| ev_등록대수 | 자치구별 전기차 등록대수 (2026년 3월) |

In [14]:
#741개 주차장 리스트
df_axis3['pklt_nm'].unique().tolist()

['15-가600구간',
 '21-37',
 'BYC',
 'KBS별관뒤 노상주차장',
 'KBS별관옆 노상주차장',
 'KBS본관 앞 노상주차장',
 'KBS본관뒤',
 'LG텔레콤',
 'MBC방송국뒤 노상주차장',
 'SJ테크노빌',
 '舊MBC뒤',
 '가든예식장 앞 노상주차장',
 '가락 1공영주차장',
 '가락 2공영주차장',
 '가로공원 지하 주차장',
 '가로공원공영주차장',
 '가마산고가밑 공영주차장',
 '가산동 공영주차장',
 '가산동 금천교 공영주차장',
 '가산디지털 서측',
 '가산디지털환승',
 '가산중뒤',
 '가산중앞',
 '가양3동 공영주차장',
 '가양3동 황금내공원길',
 '가양라이품 공영주차장',
 '가양역 공영주차장',
 '가오리 공영주차장',
 '가오천 노상공영주차장',
 '가재울 공영주차장',
 '간송옛집 공영주차장',
 '갈현1동공영주차장',
 '강남구 개포동 1-1',
 '강남대로150길',
 '강남세곡체육공원',
 '강남치매지원센터 공영주차장',
 '강남한티주차장',
 '강변(건물) 공영주차장',
 '강일동 주차장',
 '개봉1동 노외주차장',
 '개봉1동마을공동주차장',
 '개봉고가 공영주차장',
 '개운산',
 '개운산공영주차장',
 '개포동공원 공영주차장',
 '개화산역 공영주차장',
 '개화역 공영주차장',
 '거여동(기계식)',
 '건영백화점앞',
 '견인보관소',
 '견인보관소 노외',
 '경남 공영주차장',
 '경도상가앞 노상주차장',
 '경동고',
 '경동고 공영주차장',
 '경창시장 고객주차장',
 '경호빌딩앞 노상주차장',
 '고분다리 전통시장 공영주차장',
 '고분다리전통시장 주차장',
 '고척1동마을공동주차장',
 '고척근린공원지하주차장',
 '고척리본타운',
 '고척리본타운주차장',
 '곰달래 문화복지센터 공영주차장',
 '곰달래로5길 노상',
 '곰달래주차장',
 '공덕1-1 공영주차장',
 '공항2임시',
 '공항동(송정역)',
 '관악신사시장 공영주차장',
 '관악초 공영주차장',
 